# 01 OASR for Alternative Sheaves: DiscoGP with Overlap Penalty

Replicate the OASR-style three-run comparison with this repo's DiscoGP implementation: seed 42, seed 43, and an overlap-penalized seed-42 run using the seed-42 circuit as reference. The notebook loads finalized artifacts by default and can regenerate them by setting `RUN_EXPERIMENT = True`.

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from circuit_discovery.run import (
    get_compute_device,
    load_configs,
    load_model,
    load_task_dataset_from_config,
    load_circuit_map,
    evaluation_rows,
    pairwise_iou_rows,
    train_loader_from_config,
)

configs = load_configs()
print("project root:", PROJECT_ROOT)
print("device:", get_compute_device())

params = configs["notebooks"]["01_oasr_alternative_sheaves"]["hyperparams"]
artifacts = configs["artifacts"]["discogp"]["circuits"]
params


In [ ]:
# Optional regeneration. Expensive; leave disabled when browsing saved artifacts.
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    from circuit_discovery.algorithms.discogp import DiscoGP, DiscoGPConfig
    from circuit_discovery.metrics import discogp_fidelity_loss, discogp_completeness_loss
    from circuit_discovery.utils import set_seed

    output_dir = PROJECT_ROOT / configs["artifacts"]["discogp"]["root"]
    output_dir.mkdir(parents=True, exist_ok=True)
    data = load_task_dataset_from_config(params)

    warmup_e = int(0.8 * params["n_epochs_e"])
    cooldown_e = max(params["n_epochs_e"] - warmup_e, 0)

    def make_config(*, overlap_penalty: bool) -> DiscoGPConfig:
        return DiscoGPConfig(
            model_name=params["model_name"],
            prune_edges=True,
            prune_weights=False,
            n_epochs_e=params["n_epochs_e"],
            batch_size=params["batch_size"],
            lr_e=params["lr_e"],
            edge_logit_init_mean=params["edge_logit_init_mean"],
            edge_logit_init_std=params["edge_logit_init_std"],
            random_mode=params["random_mode"],
            gs_temp_edge=params["gs_temp_edge"],
            lambda_sparse_e=params["lambda_sparse_e"],
            min_times_lambda_sparse_e=params["min_times_lambda_sparse_e"],
            max_times_lambda_sparse_e=params["max_times_lambda_sparse_e"],
            n_epoch_warmup_lambda_sparse_e=warmup_e,
            n_epoch_cooldown_lambda_sparse_e=cooldown_e,
            lambda_complete_e=params["lambda_complete_e"],
            completeness_start_frac=params["completeness_start_frac"],
            lambda_overlap_e=params["lambda_overlap_e"],
            min_times_lambda_overlap_e=1.0,
            max_times_lambda_overlap_e=1.0,
            n_epoch_warmup_lambda_overlap_e=0,
            n_epoch_cooldown_lambda_overlap_e=0,
            overlap_penalty=overlap_penalty,
            tqdm_disabled=False,
        )

    for seed in params["seeds"]:
        set_seed(seed)
        model = load_model(params["model_name"])
        runner = DiscoGP(model=model, config=make_config(overlap_penalty=False))
        circuit = runner.discover_circuit(
            train_loader_from_config(data.train.dataset, params),
            fidelity_loss_fn=discogp_fidelity_loss,
            completeness_loss_fn=discogp_completeness_loss,
            finalize=True,
        )
        torch.save(
            {"circuit": circuit, "seed": seed, "algorithm": "discogp"},
            output_dir / f"seed_{seed}.pt",
        )

    reference = load_circuit_map({"seed_42": artifacts["seed_42"]})["seed_42"]
    set_seed(params["reference_seed"])
    model = load_model(params["model_name"])
    runner = DiscoGP(model=model, config=make_config(overlap_penalty=True))
    runner.load_reference_circuit(reference)
    circuit = runner.discover_circuit(
        train_loader_from_config(data.train.dataset, params),
        fidelity_loss_fn=discogp_fidelity_loss,
        completeness_loss_fn=discogp_completeness_loss,
        finalize=True,
    )
    torch.save(
        {
            "circuit": circuit,
            "seed": params["reference_seed"],
            "algorithm": "discogp",
            "overlap_reference": "seed_42",
        },
        output_dir / "seed_42_overlap_ref_seed_42.pt",
    )


In [ ]:
model = load_model(params["model_name"])
data = load_task_dataset_from_config(params)
circuits = load_circuit_map(artifacts)
print("loaded circuits:", list(circuits))
display(evaluation_rows(model, data.test, circuits))


In [ ]:
display(pairwise_iou_rows(circuits))
